# MBB intro — sportsdataverse-py

ESPN-backed NCAA men's basketball: play-by-play, schedule, teams, game rosters. The `espn_mbb_*` surface mirrors the NBA wrappers — same shape, different league.

R companion: [hoopR](https://hoopR.sportsdataverse.org) (men's basketball: NBA + NCAA). Part of the [SportsDataverse](https://py.sportsdataverse.org/docs/ecosystem).

## Setup

```sh
pip install sportsdataverse
```

In [ ]:
import polars as pl
import sportsdataverse as sdv

## Teams

In [ ]:
teams = sdv.mbb.espn_mbb_teams()
teams.shape

In [ ]:
teams.select(['team_id', 'team_location', 'team_name', 'team_abbreviation']).head()

## Schedule

In [ ]:
schedule = sdv.mbb.espn_mbb_schedule(dates=20240408)  # 2024 national championship day
schedule.select(['id', 'home_display_name', 'away_display_name', 'home_score', 'away_score']).head()

## Multi-season parquet loader

In [ ]:
schedule_2024 = sdv.mbb.load_mbb_schedule(seasons=[2024])
schedule_2024.shape

## Play-by-play — 2024 men's national championship

In [ ]:
pbp = sdv.mbb.espn_mbb_pbp(game_id=401638636)
list(pbp.keys())[:8]

In [ ]:
plays = pl.DataFrame(pbp['plays'], infer_schema_length=None)
plays.select(['period.number', 'clock.displayValue', 'text', 'scoringPlay']).head()

## Game rosters

In [ ]:
rosters = sdv.mbb.espn_mbb_game_rosters(game_id=401638636)
rosters.select(['athlete_id', 'athlete_display_name', 'team_abbreviation', 'starter']).head()

## Polars summary — teams by active status

The ESPN teams frame doesn't carry a conference column, so here's a simple grouped
count as a polars warm-up.

In [ ]:
(teams
    .group_by('team_is_active')
    .agg(pl.len().alias('teams'))
    .sort('teams', descending=True))

## Pipeline example: highest-scoring tournament games

Pull a date window covering March Madness 2024 and rank by total points.

In [ ]:
march = sdv.mbb.espn_mbb_schedule(dates='20240321-20240408')  # March Madness 2024 window
(march
    .with_columns((pl.col('home_score').cast(pl.Int64, strict=False) + pl.col('away_score').cast(pl.Int64, strict=False)).alias('total'))
    .sort('total', descending=True)
    .select(['date', 'home_display_name', 'away_display_name', 'home_score', 'away_score', 'total'])
    .head(10))

## Cross-references

- R companion: [hoopR](https://hoopR.sportsdataverse.org)
- Data source: ESPN MBB API
- Plotting: matplotlib, plotnine

## Where to go next

- API docs: `docs/docs/mbb/index.md`
- Next notebook: `07_nhl_intro.ipynb`